# aw_06_b3 — Stage B3: P1 general DPO on the Phase-1 SFT champion (Track B, §5.1/§6)

**Champion selection record (2026-08-10, protocol §6 three-stage on the V2 probes):**
transfer proxy (probe eval-ID pass_rate) 0.2567 (b1v2) vs 0.2500 (b2v2), p=0.88 → tie;
rule-OOD 0.1933 vs 0.2100, p=0.61 → tie; GPU-hours tie-break (B1 requires no RS
generation pass) → **champion = B1v2**
(`20260807-225109--b1-general-sft-v2--s42--e6e83b`, m97j/aw-runs-b1).

Cell order: `a_b3_data` (mine exact-answer pairs from the champion policy) →
`b_b3_train` (DPO, lineage-verified) → `c_b3_eval` (P1 held-out) →
`d_b3_probe` → `e_b3_probe_eval` → `x09e_run_audit` → `f_b3_analysis`.

**Stage record (CLOSED 2026-08-11):** mining run: 6041/6043 verifiable prompts,
2841 pairs (yield .4703; margin-reject 2970, no-passed 218, identical 12).
DPO run `20260811-121326--b3-general-dpo--s42--bfab0a` (rewards/accuracies .61,
margins +.076, healthy low-LR profile). P1 holdout .838 [.806,.870] (retention
vs base .828 PASS; vs b1v2 .844 n.s.). Probe transfer vs b1v2-probe: all 10
suite×metric deltas positive but NONE significant (eval-ID +.0033 p=1.0;
best rule-OOD mean_score +.0141 p=.205). x09: truncation 0.0, runaway 0.0.
**Verdict: P1 exact-answer DPO = null transfer result (no harm, no sig gain).
§6 champion among {B1v2, B2v2, B3}: proxy 3-way tie → rule-OOD tie →
GPU-hours → champion stays B1v2.** B4 (aw_07) initializes from B1v2.


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


Cloning into 'axiom-world'...
remote: Enumerating objects: 574, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 574 (delta 66), reused 71 (delta 27), pack-reused 451 (from 1)
Receiving objects: 100% (574/574), 252.43 KiB | 1.30 MiB/s, done.
Resolving deltas: 100% (298/298), done.
/content/axiom-world
Obtaining file:///content/axiom-world
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 153.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 68.1 MB/s eta 0:00:00
  Building editable for axiom-world (pyproject.toml) ... done
  

In [ ]:
# @title fetch champion — materialize the b1v2 final adapter + lineage sha
B1V2_RUN_ID = "20260807-225109--b1-general-sft-v2--s42--e6e83b"

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1V2_RUN_ID}
b1v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

import json
b1v2_sha = json.load(open(f"runs/{B1V2_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]
print(b1v2_dir, b1v2_sha)


runs/20260807-225109--b1-general-sft-v2--s42--e6e83b/artifacts/final_adapter sha256:747e57574f811274d34852f8347c47c2ff90a86186f616f796856c7f01cae21d


In [ ]:
# @title a_b3_data — mine exact-answer preference pairs from the champion (GPU)
# Same mining engine/schema as A2 (generation.pair_mining); verifier is the
# frozen ExactAnswerVerifier against gold answers in the P1 record metadata.
# Report pair_yield + decision_counts from the manifest in the checklist.
!python scripts/fetch_dataset.py \
  --repo m97j/axiom-general-posttrain \
  --path p1/v1/p1_general_sft.jsonl \
  --output data/p1/p1_general_sft.jsonl

!python scripts/mine_p1_pairs.py \
  --config configs/experiments/b3_general_dpo.yaml \
  --adapter-dir {b1v2_dir} \
  --input data/p1/p1_general_sft.jsonl \
  --num-candidates 8 --temperature 0.8 --max-new-tokens 768 --batch-size 64 \
  --output data/p1/p1_general_preference.jsonl \
  --hf-sync-repo m97j/axiom-general-posttrain --hf-path-in-repo p1/pref-v1


p1_general_sft.jsonl: 100% 5.86M/5.86M [00:00<00:00, 19.6MB/s]
fetched dataset: hf://m97j/axiom-general-posttrain/p1/v1/p1_general_sft.jsonl
revision: main
materialized: data/p1/p1_general_sft.jsonl
dataset sha256: 219e54b8baa84e6f1bf60bae85386262fe6351f204ee062743d8e1b9c4e9b2ee
DATASET_PATH=data/p1/p1_general_sft.jsonl
DATASET_SHA256=219e54b8baa84e6f1bf60bae85386262fe6351f204ee062743d8e1b9c4e9b2ee
config.json: 100% 729/729 [00:00<00:00, 8.03MB/s]
tokenizer_config.json: 100% 9.68k/9.68k [00:00<00:00, 35.1MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 76.6MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 265MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 328MB/s]
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /u

In [ ]:
# @title b_b3_train — DPO from the b1v2 parent (lineage-verified)
!python scripts/run_experiment.py \
  --config configs/experiments/b3_general_dpo.yaml \
  --parent-adapter-dir {b1v2_dir} \
  --override lineage.parent_adapter.sha256={b1v2_sha} \
  --override data.source.local_path=data/p1/p1_general_preference.jsonl \
  --hf-sync-repo m97j/aw-runs-b3


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
run_id: 20260811-121326--b3-general-dpo--s42--bfab0a
Loading weights: 100% 399/399 [00:01<00:00, 338.46it/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
[transformers] warmup_ratio is deprecated and

In [ ]:
# @title c_b3_eval — P1 held-out accuracy (vs b1v2 0.844) + per-row dump
B3_RUN_ID = "20260811-121326--b3-general-dpo--s42--bfab0a"  # <- from b_b3_train "run_id: ..."

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b3 --run-id {B3_RUN_ID}
b3_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/fetch_dataset.py \
  --repo m97j/axiom-general-posttrain \
  --path p1/v1/p1_general_holdout.jsonl \
  --output data/p1/p1_general_holdout.jsonl \
  --expected-sha256 93fb7fa354df56213b2520b5db6fbe2d4690b61f6ca1707061981ed2fae06af7

!python scripts/run_p1_eval.py \
  --config configs/experiments/b3_general_dpo.yaml \
  --adapter-dir {b3_dir} --label b3-dpo \
  --output runs/p1_eval_b3.json \
  --dump-predictions runs/p1_eval_b3_predictions.jsonl \
  --hf-sync-repo m97j/aw-runs-b3

!python scripts/x09_termination_audit.py \
  --p1-predictions runs/p1_eval_b3_predictions.jsonl --out runs/x09_drift_audit_b3.json


p1_general_holdout.jsonl: 100% 473k/473k [00:00<00:00, 26.0MB/s]
fetched dataset: hf://m97j/axiom-general-posttrain/p1/v1/p1_general_holdout.jsonl
revision: main
materialized: data/p1/p1_general_holdout.jsonl
dataset sha256: 93fb7fa354df56213b2520b5db6fbe2d4690b61f6ca1707061981ed2fae06af7
DATASET_PATH=data/p1/p1_general_holdout.jsonl
DATASET_SHA256=93fb7fa354df56213b2520b5db6fbe2d4690b61f6ca1707061981ed2fae06af7
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading weights: 100% 399/399 [00:01<00:00, 337.86it/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=

In [ ]:
# @title d_b3_probe — frozen transfer probe (200 steps) from the B3 policy
import json
b3_sha = json.load(open(f"runs/{B3_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]

!python scripts/build_training_data.py \
  --seed 1042 \
  --scenarios-per-family 400 \
  --output-dir data/train

!python scripts/run_experiment.py \
  --config configs/experiments/probe_playworld_sft.yaml \
  --parent-adapter-dir {b3_dir} \
  --override lineage.parent_adapter.repo_id=m97j/aw-runs-b3 \
  --override lineage.parent_adapter.sha256={b3_sha} \
  --override experiment_name=probe-playworld-sft-b3 \
  --hf-sync-repo m97j/aw-runs-b3-probe


{
  "seed": 1042,
  "sft_records": 2000,
  "prompt_records": 2000,
  "unsolvable_dropped": 0,
  "sft_fingerprint": "sha256:050f94b2fcddbfacc24be4fc9d843e6284ad515b837beba16808524efc8452d9",
  "prompt_fingerprint": "sha256:cc2aef0df4f5efda3db7ccfd43c76e40260935e9b7b55ee5760490e6a4304f14",
  "train_families": [
    "train-fam0",
    "train-fam1",
    "train-fam2",
    "train-fam3",
    "train-fam4"
  ],
  "eval_families_checked": [
    "eval_adversarial-fam0",
    "eval_comp_ood-fam0",
    "eval_comp_ood-fam1",
    "eval_id-fam0",
    "eval_id-fam1",
    "eval_id-fam2",
    "eval_rule_ood-fam0",
    "eval_rule_ood-fam1",
    "eval_template_ood-fam0",
    "eval_template_ood-fam1"
  ]
}
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load t

In [ ]:
# @title e_b3_probe_eval — B3 probe adapter on the frozen suites
B3_PROBE_RUN = "20260811-132220--probe-playworld-sft-b3--s42--c1d76c"  # <- from d_b3_probe

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b3-probe --run-id {B3_PROBE_RUN}
b3p_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/build_eval_suites.py --episodes-per-suite 300

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b3p_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b3-probe


eval_id: 300 episodes -> data/eval_suites/eval_id.jsonl (sha256:aceeea727d2b9eaed...)
eval_template_ood: 300 episodes -> data/eval_suites/eval_template_ood.jsonl (sha256:13580a6cbf7a4e566...)
eval_comp_ood: 300 episodes -> data/eval_suites/eval_comp_ood.jsonl (sha256:444191a244dcd77d1...)
eval_rule_ood: 300 episodes -> data/eval_suites/eval_rule_ood.jsonl (sha256:d73745108f5e7e207...)
eval_adversarial: 300 episodes -> data/eval_suites/eval_adversarial.jsonl (sha256:c73dd155acd069292...)

G3 freeze manifest -> data/eval_suites/freeze_manifest.json
Commit this manifest; training loaders must pass eval_family_ids as forbidden_family_ids (leakage gate).
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python

In [ ]:
# @title x09e_run_audit — termination regression check on the B3 probe eval (CPU)
B3_PROBE_EVAL = "20260811-133725--eval-playworld--s42--916604"  # <- eval run id from e_b3_probe_eval

!python scripts/fetch_run.py --repo m97j/aw-runs-b3-probe --run-id {B3_PROBE_EVAL} --kind eval
!python scripts/x09_termination_audit.py \
  --run-dirs runs/{B3_PROBE_EVAL} --out runs/x09_run_audit_b3.json


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0% 0/8 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.27k [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  85% 3.27k/3.85k [00:00<00:00, 16.0kB/s]

Fetching 8 files:  12% 1/8 [00:00<00:01,  4.72it/s]
Reconstructing (incomplete total...):   1% 3.85k/444k [00:00<00:27, 16.0kB/s] 
Reconstructing (incomplete total...):   1% 3.85k/445k [00:00<00:27, 16.0kB/s]
Reconstructing (incomplete total...):  50% 445k/892k [00:00<00:27, 16.0kB/s] 
Reconstructing (incomplete total...):  70% 892k/1.28M [00:00<00:00, 3.51MB/s]

Fetching 8 files:  25% 2/8 [00:00<00:00,  6.47it/s]
Reconstructing (incomplete total...):  74% 1.28M/1.74M [00:00<00:00, 3.51MB/s]
Reconstructing (incomplete total...):  79% 1.74M/2.19M [00:00<00:00, 3.51MB/s]

Fetching 8 files: 100% 8/8 [00:00<00:00, 17.75it/s]
Download complete: 100% 2.19M/2.19M [00:00<00:00, 6.06MB/s]
Reconstruction complete: 100

In [ ]:
# @title f_b3_analysis — B3 probe vs b1v2 probe (primary: does P1 DPO help transfer?)
B1V2_PROBE_EVAL = "20260810-063029--eval-playworld--s42--a7a47a"

!python scripts/fetch_run.py --repo m97j/aw-runs-b1-probe --run-id {B1V2_PROBE_EVAL} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{B3_PROBE_EVAL} --label-a b3-probe \
  --run-b runs/{B1V2_PROBE_EVAL} --label-b b1v2-probe \
  --output runs/{B3_PROBE_EVAL}/analysis_b3_vs_b1v2_probe.json --hf-sync-repo m97j/aw-runs-b3-probe


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0% 0/19 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/453k [00:00<?, ?B/s]          
Reconstructing (incomplete total...):   0% 0.00/459k [00:00<?, ?B/s]

Fetching 19 files:   5% 1/19 [00:00<00:03,  4.72it/s]
Reconstructing (incomplete total...):  50% 459k/915k [00:00<00:16, 28.3kB/s] 
Reconstructing (incomplete total...):  34% 459k/1.36M [00:00<00:31, 28.3kB/s]
Reconstructing (incomplete total...): 100% 1.36M/1.37M [00:00<00:00, 28.3kB/s]
Reconstructing (incomplete total...):  76% 1.37M/1.81M [00:00<00:15, 28.3kB/s]
Reconstructing (incomplete total...):  62% 1.37M/2.20M [00:00<00:29, 28.3kB/s]
Reconstructing (incomplete total...):  62% 1.37M/2.20M [00:00<00:29, 28.3kB/s]
Reconstructing (incomplete total...): 100% 2.20M/2.20M [00:00<00:00, 28.3kB/s]
Reconstructing (incomplete total...): 100% 2.20M/2.20M [00:00<00:00, 6.50MB/s]

Fetching 19 files:  47% 9/19 [00:00<00:0

## Stage checklist — CLOSED 2026-08-11
- [x] preference manifest recorded: 2841 pairs, yield .4703, decision_counts logged,
      fingerprint sha256:3e0f3f8d... frozen at hf dataset p1/pref-v1
- [x] b_b3_train lineage verified (parent sha256:747e5757... = b1v2 output)
- [x] P1 held-out: **b3 .838** vs b1v2 .844 (n.s., CI overlap) vs base .828 → retention PASS
      (gsm8k .8871, math .7080; trunc 3/500); x09d drift_caused_failures = 0
- [x] x09e: truncation_rate 0.0, runaway_rate 0.0 — stop behavior survives DPO
- [x] f_b3_analysis vs b1v2-probe: all deltas ≥ 0, none significant → **null result**
- [x] §6 Stage-1 record COMPLETE: base holdout .828 [.794,.860] — b1v2 +1.6pt,
      b2v2 +0.6pt, b3 +1.0pt: all pass the ≤3pt-drop hard constraint
- [x] **P1 champion = B1v2** (proxy tie b3 .260 / b1v2 .2567 p=1.0 → rule-OOD tie
      → GPU-hours: B1v2 requires no mining+DPO pass)
- [ ] (carried) B2 RS-generation manifest acceptance_rate — non-blocking (B2 not champion);
      report as descriptive statistic in the tech report if recoverable
- [x] Next stage: B4 (P1 champion → PlayWorld SFT, full A1 budget) — `aw_07_b4.ipynb`
